# 1. Exploring the data and finding how it was generated

**Competition:** [Kaggle Playground Series S6E9 — Predicting Electric Vehicle Purchases](https://www.kaggle.com/competitions/playground-series-s6e9)

The task: given 13 facts about a person (income, commute distance, age, car type, whether their
city has EV subsidies, and so on), predict whether they will buy an electric vehicle. This is a
yes/no (binary) prediction problem, scored with **ROC AUC** — a score of 0.5 means "no better than
a coin flip," 1.0 means "perfect," and in practice a well-built model on this kind of data lands
somewhere around 0.94-0.95.

Before writing a single line of model code, this notebook looks closely at the data itself. That
turns out to matter a lot here — see section 2.

## 1.1 Load the data and look at the target

In [1]:
import pandas as pd, numpy as np

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
print("train shape:", train.shape)
print("test shape :", test.shape)
print("\ntarget 'Will_Buy_EV' balance:")
print(train.Will_Buy_EV.value_counts())
print("buy rate:", round((train.Will_Buy_EV == "Yes").mean(), 4))

train shape: (668665, 15)
test shape : (286571, 14)

target 'Will_Buy_EV' balance:
Will_Buy_EV
No     551886
Yes    116779
Name: count, dtype: int64
buy rate: 0.1746


Just under 668,665 people in the training set, about 286,571 to predict on in the test set.
Only **17.5% of people buy an EV** — most people don't. This matters for modeling: a model that
just guessed "No" for everyone would already be right 82.5% of the time, so plain accuracy would
be a misleading way to judge this model. That's exactly why the competition uses ROC AUC instead
— it measures how well the model *ranks* buyers above non-buyers, regardless of how rare buying is.

## 1.2 Which columns are categories, and which are numbers?

In [1]:
def is_cat(s):
    """True for text columns. pandas >=3 uses a native 'str' dtype instead of
    'object', so we test for 'not numeric' rather than checking for 'object'."""
    return not pd.api.types.is_numeric_dtype(s)

feat = [c for c in train.columns if c not in ("id", "Will_Buy_EV")]
cats = [c for c in feat if is_cat(train[c])]
nums = [c for c in feat if c not in cats]
print("categorical columns:", cats)
print("numeric columns    :", nums)

categorical columns: ['Gender', 'City_Type', 'Current_Car_Type', 'Home_Charging_Possible', 'Subsidy_Available', 'Range_Anxiety_Level']
numeric columns    : ['Age', 'Annual_Income_USD', 'Daily_Commute_km', 'Number_of_Cars_Owned', 'Charging_Stations_Near_Home', 'Charging_Stations_Near_Work', 'Environmental_Concern_Level']


**A small but important bug worth knowing about:** the standard pandas check for a text column
is `df[col].dtype == object`. In pandas 3 (released 2025), text columns use a new native `str`
dtype instead of `object`, so that check silently returns nothing, and every "categorical" column
gets treated as if it doesn't exist. The fix is to check `pd.api.types.is_numeric_dtype(s)` instead
— that works on every pandas version. This kind of thing is easy to miss because the code doesn't
error, it just quietly does the wrong thing.

## 1.3 Buy rate by category — which factors matter most?

In [1]:
y = (train.Will_Buy_EV == "Yes").astype(int)
for c in cats:
    rates = train.groupby(c)["Will_Buy_EV"].apply(lambda s: round((s == "Yes").mean(), 3))
    print(f"{c}:")
    print(rates.to_string(), "\n")

Gender:
Gender
Female    0.178
Male      0.172
Other     0.174

City_Type:
City_Type
Rural       0.193
Suburban    0.181
Urban       0.161

Current_Car_Type:
Current_Car_Type
Hatchback    0.174
SUV          0.181
Sedan        0.172
Truck        0.156

Home_Charging_Possible:
Home_Charging_Possible
No     0.127
Yes    0.196

Subsidy_Available:
Subsidy_Available
No     0.006
Yes    0.275

Range_Anxiety_Level:
Range_Anxiety_Level
High      0.001
Low       0.189
Medium    0.042


Two columns stand out immediately: **Subsidy_Available** (people with a subsidy buy 46x more
often — 27.5% vs 0.6%) and **Range_Anxiety_Level** (people with "High" range anxiety almost never
buy — 0.1% vs 18.9% for "Low"). Gender and car type barely move the needle. This already tells us
which features the model needs to lean on.

## 2. The key finding: this is synthetic data, and it has fingerprints

Playground Series datasets are generated by a program, not collected from real surveys. That's
normal and fine — but it means the data can have patterns a real-world dataset wouldn't: exact
values that repeat far more than chance would allow, and ranges where the outcome is *always*
the same with zero exceptions. Finding these patterns and exploiting them honestly (not by
peeking at test labels, just by looking hard at the training data) is where most of the real
accuracy gain in this project came from — more than any amount of model tuning.

### 2.1 A value that appears way too often

In [1]:
inc_counts = train.Annual_Income_USD.value_counts()
print("Most repeated exact income values (count, how many rows):")
print(inc_counts.head(10).to_string())
print(f"\nincome == 30000 alone: {(train.Annual_Income_USD == 30000).sum():,} rows "
      f"({(train.Annual_Income_USD == 30000).mean():.1%} of the whole training set)")
print(f"buy rate when income == 30000: {y[train.Annual_Income_USD == 30000].mean():.4f}")
print(f"buy rate overall:              {y.mean():.4f}")

Most repeated exact income values (count, how many rows):
Annual_Income_USD
30000.0    61605
86095.0     1082
96749.0     1073
65800.0     1029
92372.0      907
84578.0      905
93957.0      852
68480.0      762
82440.0      739
84768.0      725

income == 30000 alone: 61,605 rows (9.2% of the whole training set)
buy rate when income == 30000: 0.0443
buy rate overall:              0.1746


$30,000 shows up **61,605 times** — 9.2% of the entire training set. Every other income value
shows up at most ~1,000 times. That is not how real income data looks; it's the signature of a
generator with a floor or clipping rule ("if income would go below $30,000, set it to exactly
$30,000"). And the buy rate for that group (4.4%) is noticeably different from the overall average
(17.5%) — so this isn't just a data-quality quirk, it carries real signal.

### 2.2 Ranges where the outcome has zero exceptions

In [1]:
inc = train.Annual_Income_USD.astype(float)
com = train.Daily_Commute_km.astype(float)

checks = [
    ("income >= 170,537", inc >= 170537),
    ("income in [31,004, 41,970]", (inc >= 31004) & (inc <= 41970)),
    ("commute >= 83 km", com >= 83),
    ("Environmental_Concern_Level == 1", train.Environmental_Concern_Level == 1),
]
print(f"{'rule':34s} {'rows':>8s} {'buy rate':>9s}")
for label, mask in checks:
    print(f"{label:34s} {mask.sum():8,d} {y[mask].mean():9.4f}")

rule                                   rows  buy rate
income >= 170,537                       393    1.0000
income in [31,004, 41,970]            1,257    0.0000
commute >= 83 km                        186    0.0000
Environmental_Concern_Level == 1    147,476    0.0057


Two of these are **perfectly deterministic across thousands of rows**: every single person
earning $170,537+ buys an EV (393 out of 393), and every single person earning $31,004-$41,970
does not (0 out of 1,257). Real human behavior is never this clean — somebody rich would decide
not to buy, somebody in that middle income band would buy anyway. A hard rule inside the data
generator produces exactly this pattern. These rules are free, zero-risk accuracy: once verified
on enough rows with zero exceptions, they can be applied directly to predictions with no
statistical risk at all (see notebook 3).

### 2.3 What this looks like visually

In [ ]:
from IPython.display import Image
Image("../images/income_generator_artifacts.png")

![income generator artifacts](../images/income_generator_artifacts.png)

*Top: row count per income bin on a log scale — the $30,000 spike towers over its neighbors by 2-3 orders of magnitude. Bottom: buy rate per income bin — mostly a smooth upward trend, interrupted by two hard, exact boundaries.*

## 3. Why this matters for the model

A gradient-boosted tree sees `Annual_Income_USD` as one continuous number. It can learn smooth
trends and even sharp splits on its own, but it has no special way to notice "this *exact* value
shows up 61,605 times and behaves differently from its neighbors" — that's a fact about the
*distribution* of the column, not about any one row's value in isolation, and trees don't see
distributions.

The fix, covered in notebook 2, is **target encoding**: build a feature that directly stores
"what fraction of people with this exact income value bought an EV," computed safely so it
doesn't leak the answer. That hands the model the signal directly instead of hoping thousands of
trees eventually approximate it.

**Next:** `02_feature_engineering_and_modeling.ipynb`